# Football Media Corpus — NLP Lab, Week 2

**Part 2: Our Corpus (Tasks A–H)**  
**Language:** English  
**Topic:** Football / soccer  
**Text types:** encyclopedia articles, match reports, and learning guides.

The same group corpus is used in all tasks. Source text is saved as UTF-8 `.txt` files in the `corpus` folder. Numbers, punctuation, names, and other unusual characters remain in the original text.

This notebook is part of our semester NLP project, so new football texts can be added in later weeks.

## 0. Setup

We use NLTK, spaCy, and Python's `split()` function to analyse English football texts.

In [ ]:
import importlib.util
import subprocess
import sys

packages = {
    "nltk": "nltk", "spacy": "spacy", "pandas": "pandas",
    "matplotlib": "matplotlib", "requests": "requests", "bs4": "beautifulsoup4"
}
for module, package in packages.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

import json
import re
import statistics
from pathlib import Path
from collections import Counter

import requests
import nltk
import spacy
import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
from nltk.tokenize import word_tokenize, sent_tokenize
from IPython.display import display

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

nlp = spacy.blank("en")
nlp.add_pipe("sentencizer")

def split_tokenize(text):
    return text.split()

def nltk_tokenize(text):
    return word_tokenize(text, language="english")

def spacy_tokenize(text):
    return [token.text for token in nlp(text) if not token.is_space]

tokenizers = {
    "split": split_tokenize,
    "NLTK": nltk_tokenize,
    "spaCy": spacy_tokenize
}
print("Tokenizers ready:", ", ".join(tokenizers))

**Output explanation:** The three tokenizers are ready. `split()` uses spaces, while NLTK and spaCy have their own rules for punctuation and word boundaries.

### Our eight football texts

We selected two encyclopedia articles, three football match reports, and three learning guides from Wikimedia projects. The source pages and author histories are recorded in `sources.json`. These sources have reuse licenses, so we can store and analyse their text with attribution.

The downloading step extracts visible article paragraphs from HTML. It does **not** perform NLP cleaning, lowercase conversion, or punctuation removal. Original article spelling and unusual symbols remain available for analysis.

In [ ]:
SOURCES = [
  {
    "file": "01_association_football.txt",
    "title": "Association football",
    "kind": "encyclopedia",
    "site": "Wikipedia",
    "url": "https://en.wikipedia.org/wiki/Association_football",
    "license": "CC BY-SA 4.0",
    "history": "https://en.wikipedia.org/w/index.php?title=Association_football&action=history"
  },
  {
    "file": "02_fifa_world_cup.txt",
    "title": "FIFA World Cup",
    "kind": "encyclopedia",
    "site": "Wikipedia",
    "url": "https://en.wikipedia.org/wiki/FIFA_World_Cup",
    "license": "CC BY-SA 4.0",
    "history": "https://en.wikipedia.org/w/index.php?title=FIFA_World_Cup&action=history"
  },
  {
    "file": "03_real_madrid_liverpool_2018.txt",
    "title": "European football: Real Madrid wins third consecutive Champions League beating Liverpool 3-1",
    "kind": "match report",
    "site": "Wikinews",
    "url": "https://en.wikinews.org/wiki/European_football:_Real_Madrid_wins_third_consecutive_Champions_League_beating_Liverpool_3-1",
    "license": "CC BY 2.5",
    "history": "https://en.wikinews.org/w/index.php?title=European_football:_Real_Madrid_wins_third_consecutive_Champions_League_beating_Liverpool_3-1&action=history"
  },
  {
    "file": "04_france_argentina_uruguay_portugal.txt",
    "title": "FIFA World Cup 2018 Last 16: France, Uruguay send Argentina, Portugal home",
    "kind": "match report",
    "site": "Wikinews",
    "url": "https://en.wikinews.org/wiki/FIFA_World_Cup_2018_Last_16:_France,_Uruguay_send_Argentina,_Portugal_home",
    "license": "CC BY 2.5",
    "history": "https://en.wikinews.org/w/index.php?title=FIFA_World_Cup_2018_Last_16:_France,_Uruguay_send_Argentina,_Portugal_home&action=history"
  },
  {
    "file": "05_world_cup_2018_group_stage.txt",
    "title": "FIFA World Cup 2018 day three: France, Denmark and Croatia wins three points; Argentina shares the spoils with debutants Iceland",
    "kind": "match report",
    "site": "Wikinews",
    "url": "https://en.wikinews.org/wiki/FIFA_World_Cup_2018_day_three:_France,_Denmark_and_Croatia_wins_three_points;_Argentina_shares_the_spoils_with_debutants_Iceland",
    "license": "CC BY 2.5",
    "history": "https://en.wikinews.org/w/index.php?title=FIFA_World_Cup_2018_day_three:_France,_Denmark_and_Croatia_wins_three_points;_Argentina_shares_the_spoils_with_debutants_Iceland&action=history"
  },
  {
    "file": "06_football_basics.txt",
    "title": "Football (Soccer)/The Basics",
    "kind": "learning guide",
    "site": "Wikibooks",
    "url": "https://en.wikibooks.org/wiki/Football_(Soccer)/The_Basics",
    "license": "CC BY-SA 4.0",
    "history": "https://en.wikibooks.org/w/index.php?title=Football_(Soccer)/The_Basics&action=history"
  },
  {
    "file": "07_football_history.txt",
    "title": "Football (Soccer)/The History",
    "kind": "learning guide",
    "site": "Wikibooks",
    "url": "https://en.wikibooks.org/wiki/Football_(Soccer)/The_History",
    "license": "CC BY-SA 4.0",
    "history": "https://en.wikibooks.org/w/index.php?title=Football_(Soccer)/The_History&action=history"
  },
  {
    "file": "08_football_leagues_teams.txt",
    "title": "Football (Soccer)/The Leagues and Teams",
    "kind": "learning guide",
    "site": "Wikibooks",
    "url": "https://en.wikibooks.org/wiki/Football_(Soccer)/The_Leagues_and_Teams",
    "license": "CC BY-SA 4.0",
    "history": "https://en.wikibooks.org/w/index.php?title=Football_(Soccer)/The_Leagues_and_Teams&action=history"
  }
]

CORPUS = Path("corpus")
CORPUS.mkdir(exist_ok=True)

if not Path("sources.json").exists():
    Path("sources.json").write_text(
        json.dumps(SOURCES, indent=2, ensure_ascii=False), encoding="utf-8"
    )

print("Number of selected sources:", len(SOURCES))
print("Text types:", ", ".join(sorted({item["kind"] for item in SOURCES})))

**Output explanation:** Our eight sources cover three different types of football writing. The list also tells us where every text came from.

In [ ]:
# Download only files that are missing, so we do not change saved source texts.
headers = {"User-Agent": "FootballNLPStudentCorpus/1.0 (educational coursework)"}

def article_text(html):
    soup = BeautifulSoup(html, "html.parser")
    body = (soup.select_one("#mw-content-text .mw-parser-output")
            or soup.select_one("#mw-content-text")
            or soup.select_one(".mw-parser-output"))
    if body is None:
        raise ValueError("Could not find the article text")
    for bad in body.select(
        "style, script, .mw-editsection, .navbox, .infobox, "
        ".reflist, .metadata, .thumb, .hatnote, .toc, .noprint"
    ):
        bad.decompose()
    parts = []
    for element in body.find_all(["p", "li"]):
        if element.find_parent(["table", "aside", "figure", "li"]):
            continue
        text = element.get_text(separator="", strip=False).strip()
        if len(text.split()) >= 6:
            parts.append(text)
    return "\n\n".join(parts)

for source in SOURCES:
    file = CORPUS / source["file"]
    if file.exists():
        text = file.read_text(encoding="utf-8")
        print("Already saved:", file.name, "-", len(text.split()), "words")
        continue

    try:
        response = requests.get(source["url"], headers=headers, timeout=40)
        response.raise_for_status()
        text = article_text(response.text)
    except (requests.RequestException, ValueError):
        from urllib.parse import urlparse, unquote
        page = urlparse(source["url"])
        title = unquote(page.path.split("/wiki/", 1)[-1]).replace("_", " ")
        api_url = page.scheme + "://" + page.netloc + "/w/api.php"
        response = requests.get(api_url, headers=headers, timeout=40,
                                params={"action": "parse", "page": title,
                                        "prop": "text", "format": "json", "formatversion": 2})
        response.raise_for_status()
        text = article_text(response.json()["parse"]["text"])
    if len(text.split()) < 200:
        raise ValueError(f"{file.name} has only {len(text.split())} words. Check extraction.")
    file.write_text(text + "\n", encoding="utf-8")
    print("Saved:", file.name, "-", len(text.split()), "words")

files = [CORPUS / source["file"] for source in SOURCES]
texts = [file.read_text(encoding="utf-8") for file in files]

assert len(files) >= 8
assert len(set(source["kind"] for source in SOURCES)) >= 3
assert all(len(text.split()) >= 200 for text in texts)
print("\nCorpus check passed:", len(texts), "texts,", len(set(s["kind"] for s in SOURCES)), "text types")

**Output explanation:** Every text is saved in a separate file. We confirmed that all eight files contain at least 200 words. The raw text is kept for our later NLP exercises.

## Task A — Describe the corpus

We record the file name, source, text type, number of characters, and number of whitespace tokens. Then we describe the corpus in three sentences.

In [ ]:
rows = []
for source, text in zip(SOURCES, texts):
    rows.append({
        "File": source["file"],
        "Source": source["url"],
        "Kind": source["kind"],
        "Characters": len(text),
        "Whitespace tokens": len(text.split())
    })
corpus_table = pd.DataFrame(rows)
display(corpus_table)

print("1. Our corpus has", len(texts), "English football texts.")
print("2. We use", len(set(row["Kind"] for row in rows)),
      "types of writing: encyclopedia articles, match reports and learning guides.")
print("3. In total, the texts contain",
      sum(row["Whitespace tokens"] for row in rows), "whitespace tokens.")

**Output explanation:** The table shows the size and source of each document. Our corpus contains different writing styles, which makes it useful for comparing tokenizers.

## Task B — Find difficult strings

We search for URLs, emails, hashtags, numbers, hyphenated words, apostrophes, abbreviations, and other symbols. We also identify the five most interesting cases and the text type with the most potential problems.

In [ ]:
patterns = {
    "URL": r"https?://\S+|www\.\S+",
    "email": r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+",
    "hashtag/@": r"[#@]\w+",
    "number": r"\d[\d.,:/-]*\d|\d",
    "hyphenated": r"\w+(?:-\w+)+",
    "apostrophe": r"\w+['\u2019]\w+",
    "abbreviation": r"\b[A-Z][a-z]{0,3}\.",
    "other symbol": r"[^\w\s.,;:!?'\"()\-]"
}

pattern_rows = []
cases = []
for kind, pattern in patterns.items():
    occurrences = []
    for source, text in zip(SOURCES, texts):
        for match in re.finditer(pattern, text):
            occurrences.append((source["file"], source["kind"], match.group()))
            cases.append({"category": kind, "file": source["file"],
                          "kind": source["kind"], "text": match.group()})
    freq = Counter(item[2] for item in occurrences)
    pattern_rows.append({
        "Category": kind,
        "Matches": len(occurrences),
        "Three examples": ", ".join(repr(x) for x, _ in freq.most_common(3))
    })

pattern_table = pd.DataFrame(pattern_rows)
display(pattern_table)
most_common = max(pattern_rows, key=lambda row: row["Matches"])
print("Most frequent category:", most_common["Category"],
      "-", most_common["Matches"], "matches")

**Output explanation:** Numbers, punctuation, and special formats can create several tokens instead of one. The table shows which kinds of difficult strings appear most often in our football texts.

In [ ]:
# Count matches for each type of text.
problem_by_kind = Counter()
words_by_kind = Counter()
for source, text in zip(SOURCES, texts):
    kind = source["kind"]
    words_by_kind[kind] += len(text.split())
    for pattern in patterns.values():
        problem_by_kind[kind] += len(re.findall(pattern, text))

kind_table = pd.DataFrame([
    {"Text type": kind, "Matches": problem_by_kind[kind],
     "Matches per 100 words": round(100 * problem_by_kind[kind] / words_by_kind[kind], 1)}
    for kind in sorted(words_by_kind)
])
display(kind_table)

hardest_kind = max(words_by_kind,
                   key=lambda kind: problem_by_kind[kind] / words_by_kind[kind])
print("Most potential tokenization problems per 100 words:", hardest_kind)

# Prefer five different categories, including rare but challenging formats.
order = ["URL", "email", "hashtag/@", "hyphenated", "apostrophe",
         "abbreviation", "number", "other symbol"]
hard_cases = []
used = set()
for category in order:
    choices = [c for c in cases if c["category"] == category
               and (c["file"], c["text"]) not in used]
    if choices:
        choice = max(choices, key=lambda item: len(item["text"]))
        hard_cases.append(choice)
        used.add((choice["file"], choice["text"]))
    if len(hard_cases) == 5:
        break
if len(hard_cases) < 5:
    for choice in cases:
        if (choice["file"], choice["text"]) not in used:
            hard_cases.append(choice)
            used.add((choice["file"], choice["text"]))
        if len(hard_cases) == 5:
            break

display(pd.DataFrame([{"Hard string": c["text"], "Category": c["category"],
                       "File": c["file"]} for c in hard_cases]))

**Output explanation:** Comparing matches per 100 words is fairer than comparing totals, because the files have different lengths. The five selected strings will be tested in Task E.

## Task C — Compare the tokenizers

We count the total tokens, different types, type/token ratio, and punctuation tokens for all eight texts.

In [ ]:
def token_stats(tokenizer):
    tokens = [word for text in texts for word in tokenizer(text)]
    types = len(set(word.lower() for word in tokens))
    punctuation = sum(not any(char.isalnum() for char in word) for word in tokens)
    return len(tokens), types, punctuation

stats_rows = []
for name, tokenizer in tokenizers.items():
    token_count, types_count, punct_count = token_stats(tokenizer)
    stats_rows.append({
        "Tokenizer": name, "Tokens": token_count,
        "Types": types_count,
        "Types/Tokens": round(types_count / token_count, 3),
        "Punctuation tokens": punct_count
    })
token_table = pd.DataFrame(stats_rows)
display(token_table)

print("Our choice: spaCy. It separates punctuation and many English word forms more clearly than split().")
print("We can check this choice using the difficult examples from Tasks E and F.")

**Output explanation:** The counts differ because each tokenizer uses different rules. We prefer spaCy for this English corpus, but we will also compare its decisions with real examples.

## Task D — Punctuation attached to words

We measure how many whitespace tokens begin or end with punctuation and print the fifteen most frequent examples.

In [ ]:
ws = [word for text in texts for word in text.split()]
glued = [word for word in ws if re.search(r"^[^\w\s]|[^\w\s]$", word)]
percentage = 100 * len(glued) / len(ws)
print("Whitespace tokens:", len(ws))
print("Tokens with attached punctuation:", len(glued))
print("Percentage:", round(percentage, 1), "%")
print("Top 15 examples:")
display(pd.DataFrame(Counter(glued).most_common(15), columns=["Token", "Count"]))

**Output explanation:** Many words still have commas, periods, or other punctuation attached when we use `split()`. This explains some of the differences between split(), NLTK, and spaCy.

## Task E — Test five difficult sentences

We find the sentence containing each difficult string from Task B. Then we compare how the three tokenizers split the sentence.

In [ ]:
# Find a sentence containing each difficult string.
file_to_text = {source["file"]: text for source, text in zip(SOURCES, texts)}
hard_sentences = []
for case in hard_cases:
    sentences_in_file = sent_tokenize(file_to_text[case["file"]])
    found = next((s for s in sentences_in_file if case["text"] in s), None)
    if found is None:
        # Some unusual characters can confuse sentence segmentation.
        found = next((line for line in file_to_text[case["file"]].splitlines()
                      if case["text"] in line), case["text"])
    hard_sentences.append(found)

for index, (case, sentence) in enumerate(zip(hard_cases, hard_sentences), 1):
    print("\nExample", index, "-", case["category"], repr(case["text"]))
    print("Source:", case["file"])
    print("Sentence:", sentence)
    for name, tokenizer in tokenizers.items():
        print(name, ":", tokenizer(sentence))

**Output explanation:** These five examples show how the same football sentence can be separated differently. We can now compare punctuation, hyphens, apostrophes, and numbers directly.

In [ ]:
# The comparison table includes our preferred tokenizer for each string.
# Some choices depend on the task: a score such as 3-1 can be kept together
# or split into separate numbers.
choices = {
    "URL": "spaCy", "email": "spaCy", "hashtag/@": "spaCy",
    "hyphenated": "spaCy", "apostrophe": "spaCy",
    "abbreviation": "spaCy", "number": "NLTK / spaCy",
    "other symbol": "NLTK / spaCy"
}
comparison_rows = []
for case in hard_cases:
    sample = case["text"]
    comparison_rows.append({
        "Hard string": sample, "split": str(tokenizers["split"](sample)),
        "NLTK": str(tokenizers["NLTK"](sample)),
        "spaCy": str(tokenizers["spaCy"](sample)),
        "Our choice": choices.get(case["category"], "spaCy")
    })
display(pd.DataFrame(comparison_rows))
print("Our opinion: spaCy usually gives the clearest English word boundaries.")
print("For match scores and similar numbers, the right answer depends on the analysis task.")

**Output explanation:** We choose spaCy for most examples because it gives useful tokens for English analysis. There is not always one correct split, especially for scores, abbreviations, and hyphenated forms.

## Task F — Where NLTK and spaCy disagree most

We compare the number of tokens produced by NLTK and spaCy in every sentence. We show the five sentences with the largest differences and explain the likely reason.

In [ ]:
all_sentences = [sentence for text in texts for sentence in sent_tokenize(text)]

sentence_diffs = []
for sentence in all_sentences:
    nltk_tokens = tokenizers["NLTK"](sentence)
    spacy_tokens = tokenizers["spaCy"](sentence)
    difference = abs(len(nltk_tokens) - len(spacy_tokens))
    sentence_diffs.append((difference, sentence, nltk_tokens, spacy_tokens))

sentence_diffs.sort(key=lambda item: item[0], reverse=True)
top_diffs = [item for item in sentence_diffs if item[0] > 0][:5]

for number, (difference, sentence, nltk_tokens, spacy_tokens) in enumerate(top_diffs, 1):
    print("\\nExample", number, "- Difference:", difference, "tokens")
    print("Sentence:", sentence[:500])
    print("NLTK:", nltk_tokens[:75])
    print("spaCy:", spacy_tokens[:75])
    possible_causes = []
    if re.search(r"\w-\w", sentence): possible_causes.append("hyphen")
    if re.search(r"\w['’]\w", sentence): possible_causes.append("apostrophe")
    if re.search(r"\d[.,:/-]\d", sentence): possible_causes.append("number or score")
    if re.search(r"https?://|www\.", sentence): possible_causes.append("URL")
    if not possible_causes: possible_causes.append("punctuation or abbreviations")
    print("Possible cause:", ", ".join(possible_causes))
    print("Our judgement: spaCy is usually more useful for separating English punctuation.")
    print("For scores or names, we would also check whether the original expression should stay together.")
print("\\nSentences with different token counts:",
      sum(item[0] > 0 for item in sentence_diffs))

**Output explanation:** The biggest disagreements are usually caused by punctuation or unusual forms of words. The original sentence helps us decide whether NLTK or spaCy gives the more useful result.

## Task G — Sentence lengths

We calculate the mean and median sentence length using spaCy tokens, then print the five longest sentences.

In [ ]:
sentence_lengths = [(len(tokenizers["spaCy"](sentence)), sentence)
                    for sentence in all_sentences if sentence.strip()]
lengths = [number for number, sentence in sentence_lengths]

print("Number of sentences:", len(lengths))
print("Mean length:", round(statistics.mean(lengths), 2), "tokens")
print("Median length:", statistics.median(lengths), "tokens")
print("\nFive longest sentences:")
for number, sentence in sorted(sentence_lengths, reverse=True)[:5]:
    print("\n", number, "tokens:", sentence[:500].replace("\n", " "))
    likely_sentence = sentence.strip().endswith((".", "!", "?"))
    print("Our first impression:", "Probably a real sentence" if likely_sentence
          else "Possibly a heading, list, or incomplete sentence")

**Output explanation:** The mean is the average sentence length, while the median is the middle value. Very long results may come from lists or formatting rather than normal sentences, so sentence boundaries need checking.

## Task H — A real Wikipedia paragraph

Our Wikipedia example is **2022 FIFA World Cup**, in English. We compare all three tokenizers on its introductory paragraph and save this extra text in our `corpus` folder.

In [ ]:
from urllib.parse import quote

wiki_title = "2022 FIFA World Cup"
wiki_url = "https://en.wikipedia.org/wiki/2022_FIFA_World_Cup"
summary_url = "https://en.wikipedia.org/api/rest_v1/page/summary/" + quote(wiki_title.replace(" ", "_"))

try:
    response = requests.get(summary_url, headers=headers, timeout=40)
    response.raise_for_status()
    wiki = response.json()["extract"]
except (requests.RequestException, KeyError, ValueError):
    response = requests.get(wiki_url, headers=headers, timeout=40)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    paragraphs = soup.select("#mw-content-text .mw-parser-output p")
    wiki = next((paragraph.get_text("", strip=True) for paragraph in paragraphs
                 if len(paragraph.get_text().split()) >= 25), "")

if not wiki:
    raise ValueError("Could not download the Wikipedia introductory paragraph")

wiki_file = CORPUS / "09_2022_world_cup_wikipedia_intro.txt"
wiki_file.write_text(wiki + "\n", encoding="utf-8")
print("Wikipedia article:", wiki_title)
print("Source:", wiki_url)
print("Paragraph:\n", wiki)
print("\nToken counts:")
for name, tokenizer in tokenizers.items():
    print(name, len(tokenizer(wiki)))

**Output explanation:** This is a real introductory paragraph from Wikipedia. The three methods can produce different token counts even on a short text, mainly because of punctuation and numbers.

In [ ]:
# Pick five expressions on which the tokenizers do not all agree.
wiki_examples = []
for expression in wiki.split():
    options = [tuple(tokenizer(expression)) for tokenizer in tokenizers.values()]
    if len(set(options)) > 1 and expression not in wiki_examples:
        wiki_examples.append(expression)
    if len(wiki_examples) == 5:
        break

wiki_rows = []
for expression in wiki_examples:
    wiki_rows.append({
        "Text": expression,
        "split": str(tokenizers["split"](expression)),
        "NLTK": str(tokenizers["NLTK"](expression)),
        "spaCy": str(tokenizers["spaCy"](expression)),
        "Reason": ("Attached punctuation" if expression[-1:] in ".,;:!?"
                   else "Number, date or score" if any(ch.isdigit() for ch in expression)
                   else "Apostrophe, hyphen or another rule"),
        "Our opinion": "NLTK / spaCy (punctuation separated)"
    })
display(pd.DataFrame(wiki_rows))
print("Different expressions found:", len(wiki_examples))

# Compare types of difficult strings seen in Wikipedia and our first eight files.
new_kinds = []
for category, pattern in patterns.items():
    if re.search(pattern, wiki) and not any(re.search(pattern, text) for text in texts):
        new_kinds.append(category)
print("Types of difficult strings not seen in the first eight files:",
      new_kinds if new_kinds else "None")

**Output explanation:** These examples show disagreements around actual words and punctuation from the Wikipedia paragraph. The last line checks whether the new paragraph introduces a type of difficult string we had not found before.

## Final questions

1. Which kind of text was hardest for the tokenizers?  
2. Which tokenizer would we choose, and how could we test our choice?  
3. What surprised us about tokenization in English football texts?

In [ ]:
print("1. The hardest text type per 100 words was:", hardest_kind)
print("   This type had", round(100 * problem_by_kind[hardest_kind] / words_by_kind[hardest_kind], 1),
      "potentially difficult matches for every 100 words.")
print("\n2. We choose spaCy for our English corpus.")
print("   To check it, we would manually mark the correct tokens in 20 difficult sentences")
print("   and compare them with the three tokenizers.")
print("\n3. The main surprise is that punctuation, match scores and football names")
print("   do not always split into the same tokens across different methods.")

**Output explanation:** We used the results of our corpus comparisons to choose a tokenizer. Tokenization is not just splitting at spaces, and the best choice depends on what we want to analyse.

## Corpus update note

The first eight files form our Week 2 core corpus. Task H adds a ninth Wikipedia introductory paragraph, which is kept as a separate extra text. The source links are stored in `sources.json`, and future weeks can add more documents while keeping the existing files unchanged.

The source texts belong to their respective Wikimedia contributors. Source URLs, article histories, and licenses are recorded in `sources.json`. The extracts preserve textual punctuation, but HTML page layout and image data are not included.